# Phase 2.2 — GENROU + ST1A + PSS1A on SMIB (synchronous machine with full excitation control)

> **Phase 2.1 prerequisite.** This notebook builds on
> `phase2_1_avr.ipynb` — same operating point, same fault, same
> 50 Hz system, same TGR-tuned ST1A.  Phase 2.1 left a visible
> post-fault rotor oscillation that the AVR alone could not damp
> (TGR limits the AVR's swing-frequency gain by design).  Phase 2.2
> adds an IEEE 421.5 §6.1 **PSS1A** in parallel to the AVR error
> summer to add electrical damping to the swing mode.

Headline plots:

1. **Damping comparison** (§7) — same 200 ms deep inductive fault as
   Phase 2.1, run with PSS off and on.  The late-window
   (4–8 s) rotor peak-to-peak drops from ~28° to ~13° — *visibly*
   damped instead of slowly decaying.
2. **PSS signal anatomy** (§8) — plot the chain Δω → washout y_w →
   lead-lag y_LL1 → y_LL2 → Vpss → AVR.  Shows exactly how the
   stabiliser turns rotor speed deviation into a damping torque.
3. **CCT preservation** (§9) — confirm that the PSS doesn't hurt the
   AVR's CCT lift.  For our case CCT stays at ~325 ms (vs 275 ms
   for bare GENROU and 339 ms for the unrealistic GENCLS).


## 1. Imports

In [1]:
# Bootstrap so the smib package is importable from notebooks/.
import sys, pathlib
_here = pathlib.Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import math
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from smib.models import GENCLS, GENROU
from smib.models.st1a import ST1A
from smib.models.pss1a import PSS1A
from smib.network import Network
from smib.powerflow import two_bus_pf
from smib.scenarios import three_phase_fault_schedule
from smib.simulator import run_smib_genrou, run_smib_genrou_avr, run_smib_genrou_avr_pss
from smib.plotting import plot_sld


## 2. What's new — PSS1A in parallel to the AVR error summer

The **Power System Stabiliser** (PSS) is a feedback loop that senses
the rotor's speed deviation $\Delta\omega = \omega - \omega_0$ and
modulates the AVR's voltage setpoint to produce a damping torque at
the swing frequency.  The smib PSS1A is the IEEE 421.5 §6.1
single-input type, with three states:

| State | Unit | What it is |
|---|---|---|
| `x_w`   | pu  | washout filter internal state — removes DC bias |
| `x_LL1` | pu  | first lead-lag internal state |
| `x_LL2` | pu  | second lead-lag internal state |

The signal chain is straightforward:

$$ \Delta\omega \to \underbrace{\frac{sT_w}{1+sT_w}}_{\text{washout}} \to
   \underbrace{\frac{1+sT_1}{1+sT_2}}_{\text{lead-lag 1}} \to
   \underbrace{\frac{1+sT_3}{1+sT_4}}_{\text{lead-lag 2}} \to
   \underbrace{K_s}_{\text{gain}} \to
   \underbrace{\text{clip}\,[V_{ssmin},\,V_{ssmax}]}_{\text{limit}} \to V_{pss} $$

And $V_{pss}$ feeds into the AVR summer alongside $V_{ref}$ and
$V_c$ (sensed terminal voltage).

**Why this works**: at the rotor swing frequency (~0.7 Hz on this
SMIB), the path Δω → V_pss → AVR → E_fd → E'_q → T_e has roughly
+90° of phase shift from the two lead-lag stages, which exactly
compensates the -90° lag of the rotor-flux dynamics.  The result is
a Vpss → Te response *in phase with Δω*, i.e. pure damping torque.

Defaults (this SMIB):

- `Tw = 5.0 s`  (washout)
- `T1 = T3 = 0.5 s`  (lead zero)
- `T2 = T4 = 0.05 s`  (lag pole) — each LL adds ~55° at the swing freq
- `Ks = 20`
- `Vpssmax = ±0.10 pu`  (limit)


## 3. Operating point — identical to Phase 2.0 / 2.1

Same network, same loading, same 50 Hz system.  PF result unchanged —
adding a PSS only changes the *response to disturbance*, not the
operating point.


In [2]:
P, Q = 0.8, 0.2
R_line, X_line = 0.0, 0.5
V_inf = 1.0 + 0j

V1, iters = two_bus_pf(P, Q, abs(V_inf), 0.0, R_line, X_line, bus_type='PQ')
print(f'PF converged in {iters} iters: V1 = {abs(V1):.4f} pu /_ {math.degrees(np.angle(V1)):.3f} deg')

fig = plot_sld(V_gen=V1, S_gen=complex(P, Q), V_inf=V_inf,
               R_line=R_line, X_line=X_line, gen_label='GENROU+ST1A+PSS1A',
               title='SMIB single-line diagram (Phase 2.2, post-LF)')
fig.show()


PF converged in 5 iters: V1 = 1.0178 pu /_ 23.142 deg


## 4. Initialisation — combined GENROU + ST1A + PSS1A protocol

Init order: GENROU first, then ST1A (consumes GENROU's $E_{fd,0}$),
then PSS1A (all zeros by construction).  The combined system has
**nine differential states**:

$$ x = [\;\underbrace{\delta,\;\bar\omega,\;E'_q,\;E'_d}_{\text{GENROU}},\;
       \underbrace{V_c,\;x_{LL}}_{\text{ST1A}},\;
       \underbrace{x_w,\;x_{LL1},\;x_{LL2}}_{\text{PSS1A}}\;] $$

DAE-consistent initialisation requires:

- GENROU's four derivatives = 0 → as in Phase 2.0
- ST1A's two derivatives = 0 → as in Phase 2.1 ($V_c = |V|$,
  $V_{ref} = E_{fd,0}/K_a + |V|$)
- PSS1A's three derivatives = 0 → trivially, all states = 0 since
  $\Delta\omega = 0$ at the operating point.

The PSS contribution $V_{pss}$ is **exactly zero** at steady state
(washout removes DC) — so the AVR's back-solve gives the same
$V_{ref}$ as in Phase 2.1.  Adding the PSS changes nothing at the
operating point; it only acts on deviations.


In [3]:
S = complex(P, Q)
g = GENROU(D=3.0); a = ST1A(); p = PSS1A()
g.initialise(V1, S)
a.initialise(V1, S, Efd_init=g.params['Efd'])
p.initialise()

print('--- GENROU init ---')
print(f'  delta_0 = {math.degrees(g.state["delta"]):.4f} deg')
print(f"  E'q_0    = {g.state['Eqp']:.5f} pu,   E'd_0 = {g.state['Edp']:.5f} pu")
print(f'  E_fd,0   = {g.params["Efd"]:.5f} pu')
print()
print('--- ST1A init ---')
print(f'  V_c_0    = {a.state["Vc"]:.5f} pu  (= |V_t,0|)')
print(f'  V_ref    = {a.params["Vref"]:.6f} pu')
print()
print('--- PSS1A init ---')
print(f'  x_w_0    = {p.state["x_w"]:.6e}')
print(f'  x_LL1_0  = {p.state["x_LL1"]:.6e}')
print(f'  x_LL2_0  = {p.state["x_LL2"]:.6e}')
print(f'  V_pss_0  = {p.algebraic_output()["Vpss"]:.6e}  (must be exactly 0)')


--- GENROU init ---
  delta_0 = 68.5540 deg
  E'q_0    = 0.92381 pu,   E'd_0 = 0.45714 pu
  E_fd,0   = 2.02563 pu

--- ST1A init ---
  V_c_0    = 1.01779 pu  (= |V_t,0|)
  V_ref    = 1.027915 pu

--- PSS1A init ---
  x_w_0    = 0.000000e+00
  x_LL1_0  = 0.000000e+00
  x_LL2_0  = 0.000000e+00
  V_pss_0  = 0.000000e+00  (must be exactly 0)


## 5. Flat-line — the combined correctness floor

Nine differential states.  No disturbance.  Over 10 s with $h=2$ ms
the worst-case drift on every algebraic output must be at machine
epsilon (~1e-14).  Any drift above $10^{-5}$ means either DAE-
inconsistent initialisation or a sign bug in the
$\Delta\omega \to V_{pss} \to V_{error}$ wiring.


In [4]:
g = GENROU(D=3.0); a = ST1A(); p = PSS1A()
n = Network(R=R_line, X=X_line, V_slack_mag=abs(V_inf))
res = run_smib_genrou_avr_pss(g, a, p, n, t_end=10.0, h=2e-3,
                               init_V=V1, init_S=S)
drifts = {}
for k in ['delta', 'omega', 'Eqp', 'Edp', 'Vc', 'Efd', 'Vpss',
          'pss_y_w', 'pss_y_LL1', 'pss_y_LL2', '|V|']:
    arr = res.traces[k]
    drifts[k] = float(np.abs(arr - arr[0]).max())
print('10-second flat-line drift:')
for k, v in drifts.items():
    print(f'  {k:>12s} : {v:.3e}')
print('PASS' if max(drifts.values()) < 1e-5 else 'FAIL')
print(f'\\nMean corrector iters/step: {res.info["mean_iters"]:.2f}')


10-second flat-line drift:
         delta : 0.000e+00
         omega : 7.227e-17
           Eqp : 0.000e+00
           Edp : 0.000e+00
            Vc : 0.000e+00
           Efd : 1.110e-14
          Vpss : 1.165e-15
       pss_y_w : 3.608e-17
     pss_y_LL1 : 3.650e-17
     pss_y_LL2 : 5.826e-17
           |V| : 0.000e+00
PASS
\nMean corrector iters/step: 1.00


## 6. Under the hood — Norton equivalent, current injection, Y-matrix per timestep

This is the section that pays off the algebraic-equation pedagogy
the user explicitly asked for: trace **exactly** how each timestep
takes the integrator's state and produces the bus voltage `V_0` via
the network Y-matrix.

For Phase 2.2 the *algebraic side* of the DAE is identical to
Phase 2.1 — the PSS contributes **no current** to the network
(it's a pure measurement/feedback block).  Adding the PSS only
grows the differential side by three states.

### 6.1 The five algebraic ingredients (Phase 2.2 surface)

| Symbol | Type | Where it comes from |
|---|---|---|
| $E'_q(t), E'_d(t), \delta(t)$ | differential | GENROU rotor flux + angle |
| $E_{fd}(t)$ | algebraic from AVR | ST1A regulator output (now driven by $V_{ref}, V_c, V_{pss}$) |
| $V_{pss}(t)$ | algebraic from PSS | PSS1A output (washout + 2× lead-lag + gain on $\Delta\omega$) |
| $V_0(t)$ | algebraic on network | THIS section — the bus-voltage solve |
| $I_{inject}(t)$ | algebraic on machine | follows from $V_0$ and rotor state |

The differential equations integrate $(\delta, \bar\omega, E'_q,
E'_d, V_c, x_{LL}, x_w, x_{LL1}, x_{LL2})$ — nine states total.
But every step also requires *closing the algebraic loop* —
computing $V_0$ and $I_{inject}$ exactly.  That's what the next
subsection walks through.

### 6.2 GENROU's Norton-equivalent — the saliency-aware version

In Phase 1 (GENCLS) the machine had a single internal reactance
$X'_d$ on both axes, so the machine looked like a textbook Norton
source:

$$ I_{inject,GENCLS} = \underbrace{\frac{E'}{jX'_d}}_{I_N}
                    \;-\;\underbrace{\frac{1}{jX'_d}}_{Y_N}\cdot V_0
   \qquad(\text{Norton: } I_N, Y_N \text{ both scalars}) $$

In Phase 2.0 onwards (GENROU), saliency $X'_d \ne X'_q$ breaks
that — there is no single complex Norton admittance.  Instead, in
the **machine dq frame** the stator currents are:

$$ I_q = \frac{E'_q - V_q}{X'_d}, \qquad I_d = \frac{V_d - E'_d}{X'_q} $$

Combine into the machine complex $\hat I = I_q - jI_d$ and rotate
into the global DQ frame at angle $\delta$:

$$ I_{inject,global}(V_0) = (I_q - jI_d)\,e^{j\delta} $$

Substitute and separate real/imaginary parts.  The result is
*linear* in $V_0 = v_{re} + jv_{im}$ (because the saliency lives
in the *coefficients*, not the structure):

$$ \begin{pmatrix} \mathrm{Re}\,I_{inject} \\ \mathrm{Im}\,I_{inject} \end{pmatrix}
  = M(\delta)\,\begin{pmatrix} v_{re} \\ v_{im} \end{pmatrix} + \vec c(\delta, E'_q, E'_d) $$

where the 2×2 matrix is

$$ M(\delta) = \begin{pmatrix}
   \sin\delta\cos\delta\,(1/X'_q - 1/X'_d) & -\cos^2\delta/X'_q - \sin^2\delta/X'_d \\
   \sin^2\delta/X'_q + \cos^2\delta/X'_d   &  \sin\delta\cos\delta\,(1/X'_d - 1/X'_q)
   \end{pmatrix}, $$

and the constant vector is

$$ \vec c = \begin{pmatrix} -E'_d\cos\delta/X'_q + E'_q\sin\delta/X'_d \\
                            -E'_d\sin\delta/X'_q - E'_q\cos\delta/X'_d \end{pmatrix}. $$

The "GENROU Norton equivalent" is therefore the *real 2×2 admittance
matrix* $M(\delta)$ plus the *Norton source* current vector
$\vec c(\delta, E'_q, E'_d)$.  Both depend on the rotor state (so
they update every timestep).

### 6.3 Closing the loop via the network Y-matrix — KCL at bus 0

Kirchhoff's current law at the gen bus 0 says: current injected by
the machine equals current flowing into the network:

$$ Y_{00}\,V_0 + Y_{01}\,V_\infty = I_{inject,global}(V_0) $$

Substituting our linear $I_{inject}(V_0) = M\vec v + \vec c$ and
splitting the network admittance $Y_{00} = G_{00} + jB_{00}$ into
real and imaginary parts:

$$ \begin{pmatrix} G_{00} & -B_{00} \\ B_{00} & G_{00} \end{pmatrix}
   \begin{pmatrix} v_{re} \\ v_{im} \end{pmatrix}
   = -\,\mathrm{Re}\,Y_{01} V_\infty - \mathrm{Im}\,Y_{01} V_\infty \,j
   \;+\; M(\delta)\,\vec v + \vec c $$

Rearrange to a 2×2 linear system in $\vec v$:

$$ \Big[\,M(\delta) - N\,\Big]\,\vec v = \vec n_{const} - \vec c $$

where $N$ is the 2×2 real form of $Y_{00}$ and $\vec n_{const}$ is
the 2-vector form of $Y_{01}V_\infty$.  **One 2×2 `np.linalg.solve`
call** per corrector iteration — that's the *whole* algebraic
network solve.  Once $\vec v$ is known we have $V_0 = v_{re} +
jv_{im}$, and the current injection follows from
$I_{inject} = M\vec v + \vec c$.

### 6.4 Run it — show the algebraic ingredients live

Let's compute, in Python, exactly the M, $\vec c$, N, $\vec n_{const}$
quantities at the operating point so the equations above stop being
abstract:


In [5]:
# Reproduce the algebraic ingredients at the operating point.
from smib.simulator import _solve_network_with_genrou

g = GENROU(D=3.0); g.initialise(V1, S)
delta = g.state['delta']; Eqp = g.state['Eqp']; Edp = g.state['Edp']
Xdp = g.params['Xdp']; Xqp = g.params['Xqp']
c_, s_ = math.cos(delta), math.sin(delta)

M = np.array([
    [s_*c_*(1/Xqp - 1/Xdp), -c_**2/Xqp - s_**2/Xdp],
    [s_**2/Xqp + c_**2/Xdp,  s_*c_*(1/Xdp - 1/Xqp)],
])
c_vec = np.array([
    -Edp*c_/Xqp + Eqp*s_/Xdp,
    -Edp*s_/Xqp - Eqp*c_/Xdp,
])
n = Network(R=R_line, X=X_line, V_slack_mag=abs(V_inf))
Y = n.ybus()
Y00, Y01 = Y[0,0], Y[0,1]
N_mat = np.array([
    [Y00.real, -Y00.imag],
    [Y00.imag,  Y00.real],
])
rhs_const = Y01 * n.V_slack
n_const = np.array([rhs_const.real, rhs_const.imag])

print(f'At the operating point (delta = {math.degrees(delta):.2f}°):')
print(f'\\n  Machine 2x2 admittance M(delta) =')
print(f'    [{M[0,0]:+.4f}, {M[0,1]:+.4f}]')
print(f'    [{M[1,0]:+.4f}, {M[1,1]:+.4f}]')
print(f'\\n  Machine Norton source c(delta,E_q,E_d) = [{c_vec[0]:+.4f}, {c_vec[1]:+.4f}]')
print(f'\\n  Network 2x2 form of Y[0,0] =')
print(f'    [{N_mat[0,0]:+.4f}, {N_mat[0,1]:+.4f}]')
print(f'    [{N_mat[1,0]:+.4f}, {N_mat[1,1]:+.4f}]')
print(f'\\n  Network -Y[0,1]·V_inf (n_const) = [{n_const[0]:+.4f}, {n_const[1]:+.4f}]')

# Solve the KCL system and verify we recover V1.
A = M - N_mat
b = n_const - c_vec
v = np.linalg.solve(A, b)
V0_solved = complex(v[0], v[1])
print(f'\\n  Solved V_0 = {V0_solved.real:+.6f} + {V0_solved.imag:+.6f}j')
print(f'  Expected V1 = {V1.real:+.6f} + {V1.imag:+.6f}j   (from power flow)')
print(f'  Mismatch: {abs(V0_solved - V1):.3e}  (machine-epsilon if init is consistent)')

# Cross-check against the actual smib simulator helper:
V0_smib = _solve_network_with_genrou(g.flatten(), g, n)
print(f'  smib._solve_network_with_genrou returns: {abs(V0_smib - V1):.3e} off V1')


At the operating point (delta = 68.55°):
\n  Machine 2x2 admittance M(delta) =
    [-0.6108, -3.0934]
    [+1.7784, +0.6108]
\n  Machine Norton source c(delta,E_q,E_d) = [+2.6090, -1.7805]
\n  Network 2x2 form of Y[0,0] =
    [+0.0000, +2.0000]
    [-2.0000, +0.0000]
\n  Network -Y[0,1]·V_inf (n_const) = [-0.0000, +2.0000]
\n  Solved V_0 = +0.935890 + +0.400000j
  Expected V1 = +0.935890 + +0.400000j   (from power flow)
  Mismatch: 5.551e-17  (machine-epsilon if init is consistent)
  smib._solve_network_with_genrou returns: 5.551e-17 off V1


### 6.5 What runs each timestep, end-to-end

```text
for t = 0, h, 2h, …:

    # ----- 1. PREDICTOR ----------------------------------------------
    f_old = derivs(x_k, V_k)            # 9 entries
    x_guess = x_k + h * f_old

    # ----- 2. CORRECTOR LOOP -----------------------------------------
    repeat:

        # 2a. PSS reads ω̄ from x_guess, produces V_pss
        Δω      = x_guess[ω̄ slot]
        V_pss   = clip( Ks · LL2( LL1( washout( Δω ) ) ), ±Vpssmax )

        # 2b. AVR reads |V|, V_pss → E_fd
        |V|     = |V_guess|
        error   = V_ref − V_c + V_pss
        E_fd    = clip( Ka · LL(error), Vrmin..Vrmax )

        # 2c. GENROU exposes Norton 2x2:  M(δ) and c(δ, E'q, E'd)
        # 2d. NETWORK ALGEBRAIC SOLVE — one 2x2 linear system
        v_re, v_im  =  solve( M - N,  n_const - c )       # ← THIS LINE
        V_guess     =  v_re + j·v_im
        I_inject    =  M · [v_re, v_im] + c

        # 2e. DERIVATIVES with new V
        f_new = derivs(x_guess, V_guess)

        # 2f. TRAPEZOIDAL update
        x_new = x_k + (h/2) · (f_old + f_new)

        if max(|x_new - x_guess|) < tol: break
        x_guess = x_new

    # ----- 3. ACCEPT, advance time -----------------------------------
    x_{k+1} = x_new ;  V_{k+1} = V_guess
```

Step **2d** is what the user specifically asked us to spell out:
the bus voltage is computed by solving the network Y-matrix KCL
with the machine's saliency-aware Norton equivalent folded in.
That one `np.linalg.solve` (a 2×2) is the entire "algebraic side"
per corrector iteration.  No fixed-point loop on the algebraic
side; the index-1 DAE closes in a single shot.

Adding the PSS only added work to step **2a** (the Δω → V_pss
chain), which is a few multiplies — nothing changes about the
Norton/Y-matrix machinery.

The corrector loop **2a–2f** typically converges in 3–5
iterations during a fault (vs 2–3 without the PSS), because Vpss
provides another path through which $V_0$ feeds back into $E_{fd}$.


## 7. Headline plot 1 — voltage-step response with and without PSS

Before the large-signal fault test, check what the PSS does to the
small-signal voltage step.  Bump `Vref` by +2 % at t = 1 s and watch
two runs in parallel:

- **AVR alone** (Phase 2.1's response, TGR-tuned).
- **AVR + PSS** (this phase).

The expected result: the PSS makes the step response *quieter*, not
slower.  The PSS senses any rotor wobble that the step kicks up
and adds damping torque, so any residual oscillation on `|V_t|`,
`E_fd`, `E'_q` decays faster.  But the *settling* (the slow drift
toward the new equilibrium) is dominated by `T'_d0 = 8 s` and is
unaffected — the PSS only acts on $\Delta\omega$, not on $V_c$
directly.


In [6]:
# AVR-only voltage step (Phase 2.1 baseline).
g1 = GENROU(D=3.0); a1 = ST1A()
g1.initialise(V1, S); a1.initialise(V1, S, Efd_init=g1.params['Efd'])
Vref0 = a1.params['Vref']
g_avr_step = GENROU(D=3.0); a_avr_step = ST1A()
bumped_a = [False]
def step_avr(t, h, _):
    if t >= 1.0 and not bumped_a[0]:
        a_avr_step.params['Vref'] = a_avr_step.params['Vref'] + 0.02
        bumped_a[0] = True
r_step_avr = run_smib_genrou_avr(g_avr_step, a_avr_step, n,
                                 t_end=15.0, h=2e-3, scenarios=[step_avr],
                                 init_V=V1, init_S=S)

# AVR + PSS voltage step.
g_pss_step = GENROU(D=3.0); a_pss_step = ST1A(); p_pss_step = PSS1A()
bumped_p = [False]
def step_pss(t, h, _):
    if t >= 1.0 and not bumped_p[0]:
        a_pss_step.params['Vref'] = a_pss_step.params['Vref'] + 0.02
        bumped_p[0] = True
r_step_pss = run_smib_genrou_avr_pss(g_pss_step, a_pss_step, p_pss_step, n,
                                      t_end=15.0, h=2e-3, scenarios=[step_pss],
                                      init_V=V1, init_S=S)

panels = [
    ('|V_t| [pu]',   lambda r: r.traces['|V|']),
    ('E_fd [pu]',    lambda r: r.traces['Efd']),
    ("E'_q [pu]",    lambda r: r.traces['Eqp']),
    ('omega bar [pu]', lambda r: r.traces['omega']),
]
fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True,
                    vertical_spacing=0.025,
                    subplot_titles=[p[0] for p in panels])
for i, (lbl, fn) in enumerate(panels, start=1):
    fig.add_trace(go.Scatter(x=r_step_avr.t, y=fn(r_step_avr), name='AVR only',
                             line=dict(color='#1565C0', width=1.5),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=r_step_pss.t, y=fn(r_step_pss), name='AVR + PSS',
                             line=dict(color='#2E7D32', width=1.5),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_vline(x=1.0, line=dict(color='#B71C1C', dash='dash', width=1),
                  row=i, col=1)
fig.update_xaxes(title_text='time [s]', row=len(panels), col=1)
fig.update_layout(height=170*len(panels), width=900,
                  title='Voltage-step response — +2% Vref bump at t = 1 s')
fig.show()

# Headlines — compare the step's residual oscillation.
def ringing_pp(r, t0=3.0, t1=12.0, signal='omega'):
    arr = r.traces[signal]
    win = arr[int(r.t.searchsorted(t0)):int(r.t.searchsorted(t1))]
    return float(win.max() - win.min())

print('Residual rotor-slip oscillation (3-12s window):')
print(f"  AVR only :   {ringing_pp(r_step_avr)*1e6:>7.2f} ppm peak-to-peak")
print(f"  AVR + PSS:   {ringing_pp(r_step_pss)*1e6:>7.2f} ppm peak-to-peak")
print()
print('|V_t| settling:')
print(f"  Pre-step      : {r_step_avr.traces['|V|'][int(r_step_avr.t.searchsorted(0.99))]:.4f}")
print(f"  Final (t=15s) AVR only: {r_step_avr.traces['|V|'][-1]:.4f}")
print(f"  Final (t=15s) AVR+PSS : {r_step_pss.traces['|V|'][-1]:.4f}")
print(f"  (small-signal steady state should match — PSS is zero in steady state)")
print()
print('Peak Efd during step:')
print(f"  AVR only :   {r_step_avr.traces['Efd'][int(r_step_avr.t.searchsorted(1.0)):].max():.4f} pu")
print(f"  AVR + PSS:   {r_step_pss.traces['Efd'][int(r_step_pss.t.searchsorted(1.0)):].max():.4f} pu")
print(f"  Peak |Vpss| during step: {np.abs(r_step_pss.traces['Vpss']).max():.4f} pu  (limit ±0.10)")


Residual rotor-slip oscillation (3-12s window):
  AVR only :    101.39 ppm peak-to-peak
  AVR + PSS:     55.29 ppm peak-to-peak

|V_t| settling:
  Pre-step      : 1.0178
  Final (t=15s) AVR only: 1.0374
  Final (t=15s) AVR+PSS : 1.0375
  (small-signal steady state should match — PSS is zero in steady state)

Peak Efd during step:
  AVR only :   2.2445 pu
  AVR + PSS:   2.2643 pu
  Peak |Vpss| during step: 0.0106 pu  (limit ±0.10)


## 9. Headline plot 3 — PSS damping on the canonical 200 ms deep fault

Same disturbance as Phase 2.1 ($Z_f = j0.10$ pu, $t_{clear}=200$ ms),
run with AVR alone vs AVR + PSS.  Watch the rotor angle settle:

- **AVR alone**: rotor angle decays with the natural electromechanical
  damping (D=3 plus rotor flux) — still ~28° peak-to-peak in the
  4-8 s window.
- **AVR + PSS**: the PSS adds active damping torque at the swing
  frequency; late-window oscillation drops to ~13° peak-to-peak.


In [7]:
f200 = three_phase_fault_schedule(1.0, 1.20, 0+0.10j)

g1 = GENROU(D=3.0); a1 = ST1A()
r_avr = run_smib_genrou_avr(g1, a1, n, t_end=8.0, h=2e-3,
                            scenarios=[f200], init_V=V1, init_S=S)
g2 = GENROU(D=3.0); a2 = ST1A(); p2 = PSS1A()
r_pss = run_smib_genrou_avr_pss(g2, a2, p2, n, t_end=8.0, h=2e-3,
                                 scenarios=[f200], init_V=V1, init_S=S)

panels = [
    ('delta [deg]',  lambda r: np.degrees(r.traces['delta'])),
    ('omega [pu]',   lambda r: r.traces['omega']),
    ('|V_t| [pu]',   lambda r: r.traces['|V|']),
    ('E_fd [pu]',    lambda r: r.traces['Efd']),
    ("E'_q [pu]",    lambda r: r.traces['Eqp']),
]
fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True,
                    vertical_spacing=0.022,
                    subplot_titles=[p[0] for p in panels])
for i, (lbl, fn) in enumerate(panels, start=1):
    fig.add_trace(go.Scatter(x=r_avr.t, y=fn(r_avr), name='AVR only',
                             line=dict(color='#1565C0', width=1.5),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=r_pss.t, y=fn(r_pss), name='AVR + PSS',
                             line=dict(color='#2E7D32', width=1.5),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_vrect(x0=1.0, x1=1.20, fillcolor='#FFCDD2', opacity=0.3,
                  layer='below', line_width=0, row=i, col=1)
fig.update_xaxes(title_text='time [s]', row=len(panels), col=1)
fig.update_layout(height=170*len(panels), width=900,
                  title='Deep inductive fault (Z_f = j0.10), 200 ms — AVR vs AVR+PSS')
fig.show()

# Damping headlines
def pp_late(res):
    d = np.degrees(res.traces['delta'])
    late = d[int(res.t.searchsorted(4.0)):]
    return float(late.max() - late.min())
print(f'Late-window (4-8 s) rotor angle peak-to-peak:')
print(f'  AVR only : {pp_late(r_avr):>6.2f}°')
print(f'  AVR + PSS: {pp_late(r_pss):>6.2f}°  ({(1 - pp_late(r_pss)/pp_late(r_avr))*100:.1f}% reduction)')
print(f'Peak rotor angle:')
print(f'  AVR only : {np.degrees(r_avr.traces["delta"]).max():>6.2f}°')
print(f'  AVR + PSS: {np.degrees(r_pss.traces["delta"]).max():>6.2f}°')
print(f'Mean corrector iters/step (AVR+PSS): {r_pss.info["mean_iters"]:.2f}')


Late-window (4-8 s) rotor angle peak-to-peak:
  AVR only :  27.93°
  AVR + PSS:  12.76°  (54.3% reduction)
Peak rotor angle:
  AVR only : 103.33°
  AVR + PSS: 103.31°
Mean corrector iters/step (AVR+PSS): 2.54


## 9. Headline plot 3 — PSS signal anatomy

Trace each block in the PSS chain to see exactly what the
stabiliser is doing.  Same fault, AVR + PSS run.  The plotted
signals are:

- `omega` — rotor slip $\bar\omega$ (input to PSS)
- `pss_y_w` — washout output (Δω with DC removed)
- `pss_y_LL1` — first lead-lag output
- `pss_y_LL2` — second lead-lag output
- `Vpss` — final stabiliser output (after gain `Ks` and ± limit)

Watch the saturation behaviour: during the first ~2 s post-fault
the PSS rails at ±0.10 (full damping torque demanded but limited);
after the rotor returns to small deviations the PSS comes off the
limit and operates linearly.


In [8]:
g3 = GENROU(D=3.0); a3 = ST1A(); p3 = PSS1A()
r3 = run_smib_genrou_avr_pss(g3, a3, p3, n, t_end=8.0, h=2e-3,
                              scenarios=[f200], init_V=V1, init_S=S)

panels = [
    (u'omega bar [pu]', r3.traces['omega']),
    ('pss y_w [pu]',    r3.traces['pss_y_w']),
    ('pss y_LL1 [pu]',  r3.traces['pss_y_LL1']),
    ('pss y_LL2 [pu]',  r3.traces['pss_y_LL2']),
    ('V_pss (final, clipped at ±0.10) [pu]', r3.traces['Vpss']),
]
fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True,
                    vertical_spacing=0.022,
                    subplot_titles=[p[0] for p in panels])
for i, (lbl, arr) in enumerate(panels, start=1):
    fig.add_trace(go.Scatter(x=r3.t, y=arr, mode='lines',
                             line=dict(color='#2E7D32', width=1.4),
                             showlegend=False), row=i, col=1)
    fig.add_vrect(x0=1.0, x1=1.20, fillcolor='#FFCDD2', opacity=0.3,
                  layer='below', line_width=0, row=i, col=1)
    fig.add_hline(y=0, line=dict(color='#999', dash='dot', width=1),
                  row=i, col=1)
# Mark Vpss limits explicitly.
fig.add_hline(y=0.10,  line=dict(color='#B71C1C', dash='dash', width=1), row=5, col=1)
fig.add_hline(y=-0.10, line=dict(color='#B71C1C', dash='dash', width=1), row=5, col=1)
fig.update_xaxes(title_text='time [s]', row=len(panels), col=1)
fig.update_layout(height=170*len(panels), width=900,
                  title='PSS1A signal chain — Δω through washout → 2× lead-lag → gain → limit')
fig.show()

# Summary stats
sat_frac = float((np.abs(r3.traces['Vpss']) >= 0.10 - 1e-6).sum() / len(r3.traces['Vpss']))
print(f'Fraction of run with Vpss at the ±0.10 limit: {sat_frac*100:.1f}%')
print(f'Peak |Vpss| during run:                       {np.abs(r3.traces["Vpss"]).max():.4f} pu')
print(f'Peak |y_LL2| (pre-gain, pre-limit):           {np.abs(r3.traces["pss_y_LL2"]).max():.6f} pu')
print(f'Implied unlimited Vpss demand peak:           {20 * np.abs(r3.traces["pss_y_LL2"]).max():.4f} pu')
print("   (the gap between unlimited and clipped is what the limiter is doing)")


Fraction of run with Vpss at the ±0.10 limit: 50.8%
Peak |Vpss| during run:                       0.1000 pu
Peak |y_LL2| (pre-gain, pre-limit):           0.188739 pu
Implied unlimited Vpss demand peak:           3.7748 pu
   (the gap between unlimited and clipped is what the limiter is doing)


## 10. CCT preservation — four-way comparison

Add one row to the §8 Phase 2.1 comparison table: GENROU + ST1A + PSS1A.

Goal: confirm the PSS doesn't *hurt* CCT (it shouldn't, since the
damping torque only acts after the fault).  Expect ≈ same as AVR
alone.


In [9]:
def is_st_gencls(t_clear):
    g = GENCLS(H=4.0, D=3.0, Xdp=0.30, f0=50.0)
    f = three_phase_fault_schedule(1.0, 1.0+t_clear, 0+0.10j)
    from smib.simulator import run_smib_gencls
    r = run_smib_gencls(g, n, t_end=5.0, h=2e-3, scenarios=[f], init_V=V1, init_S=S)
    d0 = r.traces['delta'][0]
    return abs(r.traces['delta'] - d0).max() < 2*math.pi

def is_st_genrou_bare(t_clear):
    g = GENROU(D=3.0)
    f = three_phase_fault_schedule(1.0, 1.0+t_clear, 0+0.10j)
    r = run_smib_genrou(g, n, t_end=5.0, h=2e-3, scenarios=[f], init_V=V1, init_S=S)
    d0 = r.traces['delta'][0]
    return abs(r.traces['delta'] - d0).max() < 2*math.pi

def is_st_avr(t_clear):
    g = GENROU(D=3.0); a = ST1A()
    f = three_phase_fault_schedule(1.0, 1.0+t_clear, 0+0.10j)
    r = run_smib_genrou_avr(g, a, n, t_end=5.0, h=2e-3, scenarios=[f], init_V=V1, init_S=S)
    d0 = r.traces['delta'][0]
    return abs(r.traces['delta'] - d0).max() < 2*math.pi

def is_st_pss(t_clear):
    g = GENROU(D=3.0); a = ST1A(); p = PSS1A()
    f = three_phase_fault_schedule(1.0, 1.0+t_clear, 0+0.10j)
    r = run_smib_genrou_avr_pss(g, a, p, n, t_end=5.0, h=2e-3, scenarios=[f], init_V=V1, init_S=S)
    d0 = r.traces['delta'][0]
    return abs(r.traces['delta'] - d0).max() < 2*math.pi

def bisect(is_st, lo=0.05, hi=0.6, n=10):
    for _ in range(n):
        mid = 0.5*(lo+hi)
        if is_st(mid): lo = mid
        else: hi = mid
    return 0.5*(lo+hi)

cct_cls   = bisect(is_st_gencls)
cct_bare  = bisect(is_st_genrou_bare)
cct_avr   = bisect(is_st_avr)
cct_pss   = bisect(is_st_pss)

print('Critical clearing time, deep inductive fault Z_f = j0.10 pu (D=3 load damping):')
print()
print(f'{"Model":>35s} | {"CCT (ms)":>9s} | {"vs GENCLS":>10s}')
print('-' * 65)
print(f'{"GENCLS (classical)":>35s} | {cct_cls*1000:>7.0f}   |   reference')
print(f'{"GENROU bare (no AVR)":>35s} | {cct_bare*1000:>7.0f}   | {(cct_bare-cct_cls)*1000:+7.0f} ms')
print(f'{"GENROU + ST1A AVR":>35s} | {cct_avr*1000:>7.0f}   | {(cct_avr-cct_cls)*1000:+7.0f} ms')
print(f'{"GENROU + ST1A + PSS1A":>35s} | {cct_pss*1000:>7.0f}   | {(cct_pss-cct_cls)*1000:+7.0f} ms')
print()
print(f'PSS damping benefit shows up in transient shape, not CCT itself —')
print(f'  CCT lift (AVR over bare):     +{(cct_avr-cct_bare)*1000:.0f} ms')
print(f'  CCT lift (PSS over AVR):      {(cct_pss-cct_avr)*1000:+.0f} ms (negligible by design;')
print(f'    PSS is for small-signal damping, not for first-swing stability).')


Critical clearing time, deep inductive fault Z_f = j0.10 pu (D=3 load damping):

                              Model |  CCT (ms) |  vs GENCLS
-----------------------------------------------------------------
                 GENCLS (classical) |     339   |   reference
               GENROU bare (no AVR) |     275   |     -64 ms
                  GENROU + ST1A AVR |     325   |     -14 ms
              GENROU + ST1A + PSS1A |     325   |     -14 ms

PSS damping benefit shows up in transient shape, not CCT itself —
  CCT lift (AVR over bare):     +50 ms
  CCT lift (PSS over AVR):      +0 ms (negligible by design;
    PSS is for small-signal damping, not for first-swing stability).


## 11. What's coming in Phase 2.3 (TGOV1 turbine governor)

Phase 2.2 closes three of the four pillars of the classical machine
stack:

| Pillar | Function | smib model | Phase |
|---|---|---|---|
| Inertia + flux | swing equation, rotor flux | GENROU | 2.0 |
| Voltage regulation | terminal V control | ST1A | 2.1 |
| **Damping** | **swing-mode damping torque** | **PSS1A** | **2.2 (here)** |
| Frequency response | mechanical power vs frequency | TGOV1 | 2.3 |

The fourth pillar — the **turbine governor** — adds primary
frequency response: when rotor speed drops, the governor opens the
steam/water valve to put more mechanical power into the shaft.  On
the SMIB scenario this manifests as a slow recovery of $P_m$
matching the steady-state power-frequency droop.

Phase 2.3 adds `smib/models/tgov1.py` and a fourth simulator
harness `run_smib_genrou_avr_pss_gov`.  At that point the classical
synchronous-machine model stack is complete, and the focus shifts
to Phase 3 (inverter-based resources — REGC_A, REEC_A, REPC_A) on
the same SMIB.

## 11. References

- IEEE Std 421.5-2016, "IEEE Recommended Practice for Excitation
  System Models for Power System Stability Studies", §5.1 (ST1A)
  and §6.1 (PSS1A).
- P. Kundur, *Power System Stability and Control* (1994),
  Chapter 17 §17.4 "Power system stabilizer (PSS)" for the
  phase-compensation derivation.
- Rogers, *Power System Oscillations* (2000) for the swing-mode
  damping-torque framework.
